In [ ]:
# Scenario: Customer Support Chatbot Workflow
# Imagine a company wants to build a chatbot that helps customers with quick answers. The workflow is modeled as a graph of states:

# - State Definition (BotState)
# - The chatbot keeps track of:
# - The question asked by the customer.
# - The answer generated.
# - The history of all past questions.
# - Think of this as the chatbot’s notebook where it records the conversation.

# - Nodes (Functions)
# - get_answer:
# When a customer asks, “What are your store hours?”, the chatbot looks at the question and generates a placeholder answer like “Answer to: What are your store hours?”.
# It also adds the question to the history log.
# - format_output:
# Before sending the response back to the customer, the chatbot reformats it into a friendly style:
# “Bot says: Answer to: What are your store hours?”

# - Graph Flow
# - The chatbot starts at the get_answer node (entry point).
# - Once the answer is generated, it flows to the format_output node.
# - Finally, the conversation ends at END, meaning the chatbot has
#  delivered its response.


from langgraph.graph import StateGraph, END
from typing import TypedDict

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    # In real app: call LLM here
    ans = f"Answer to: {q}"
    return {"answer": ans,
            "history": state["history"] + [q]}

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)


In [ ]:
# Scenario: Customer Support Chatbot (Question-Based)
# Imagine a company has deployed a chatbot that answers customer
#  questions by calling the Groq API. The workflow is modeled as a
#  graph of states, where each customer query flows through nodes until
#   a response is delivered.

# 1. State Definition
# The chatbot maintains a notebook-like state:
# - question → The customer’s query.
# - answer → The response generated by Groq.
# - history → A log of all past questions.


from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class BotState(TypedDict):
    question: str
    answer: str
    history: list

# 2. Define Nodes (functions)
def get_answer(state: BotState):
    q = state["question"]
    groq_api_key = userdata.get('api_key')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    # Call Groq API
    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            # IMPORTANT: The list of supported models by Groq API changes frequently.
            # Please refer to the official Groq documentation (https://console.groq.com/docs/models)
            # to find a currently active and supported model name and replace 'YOUR_GROQ_MODEL_NAME_HERE' below.
            # Examples of often available models include 'llama3-8b-8192' or 'llama3-70b-8192',
            # but these can also become decommissioned.
            "model": "llama-3.1-8b-instant",   # Changed to a currently active model
            "messages": [{"role": "user", "content": q}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: 'choices' key missing or empty. Full response: {response_json}")

    # Extract answer from Groq response
    ans = response_json["choices"][0]["message"]["content"]

    return {
        "answer": ans,
        "history": state["history"] + [q]
    }

def format_output(state: BotState):
    return {"answer": f"Bot says: {state['answer']}"}

# 3. Build the Graph
graph = StateGraph(BotState)
graph.add_node("get_answer", get_answer)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("get_answer")
graph.add_edge("get_answer", "format")
graph.add_edge("format", END)

# 5. Example Run
if __name__ == "__main__":
    # Initial state
    state = {"question": "What are your store hours?", "answer": "", "history": []}

    # Run the graph
    app = graph.compile()
    result = app.invoke(state)

    print(result["answer"])



Bot says: I'm a large language model, I don't have a physical store. I exist solely as a digital entity, so I don't have store hours. I'm available 24/7 to assist you with any questions or tasks you may have.


In [ ]:
# Scenario: AI-Powered Study Assistant (Flashcard-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each learner:
# - topic → The subject the learner is studying (e.g., "Photosynthesis").
# - flashcard → A generated Q&A card created by Groq (question on one side, answer on the other).
# - progress → A log of all past flashcards attempted, including whether the learner got them correct or not.

# 2. Workflow (Graph of States)
# Each learner interaction flows through nodes until a flashcard is delivered:
# - Input Node
# - Learner provides a topic or asks for practice (e.g., "Test me on cell biology").
# - State updates: topic = "cell biology"
# - Generation Node (Groq API)
# - Groq generates a flashcard:
# - flashcard.question = "What organelle is known as the powerhouse of the cell?"
# - flashcard.answer = "Mitochondria"
# - Response Node
# - Assistant presents the flashcard question to the learner.
# - Evaluation Node
# - Learner responds with their answer.
# - Assistant checks correctness and updates progress.
# - History Node
# - Logs the flashcard attempt:
# - progress = [{question, learner_answer, correct/incorrect}]

from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict

# 1. Define State
class StudyState(TypedDict):
    topic: str
    flashcard: Dict[str, str]
    learner_answer: str
    progress: List[Dict[str, str]]
    answer: str


# 2. Define Nodes

def generate_flashcard(state: StudyState):
    topic = state["topic"]
    return {
        "flashcard": {
            "question": f"What is a key concept of {topic}?",
            "answer": f"Key concept of {topic}"
        }
    }


def evaluate(state: StudyState):
    correct = state["flashcard"]["answer"]
    user_ans = state["learner_answer"]

    result = "correct" if user_ans.lower() in correct.lower() else "incorrect"

    return {
        "answer": result,
        "progress": state["progress"] + [{
            "question": state["flashcard"]["question"],
            "learner_answer": user_ans,
            "result": result
        }]
    }


def format_output(state: StudyState):
    return {
        "answer": f"Result: {state['answer']}"
    }


# 3. Build Graph
graph = StateGraph(StudyState)

graph.add_node("generate", generate_flashcard)
graph.add_node("evaluate", evaluate)
graph.add_node("format", format_output)


# 4. Add Edges
graph.set_entry_point("generate")
graph.add_edge("generate", "evaluate")
graph.add_edge("evaluate", "format")
graph.add_edge("format", END)


In [ ]:
#WITH API KEY
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
import requests
from google.colab import userdata

# 1. Define State
class StudyState(TypedDict):
    topic: str
    flashcard: Dict[str, str]
    learner_answer: str
    answer: str
    progress: List[Dict[str, str]]

# 2. Define Nodes (functions)

def generate_flashcard(state: StudyState):
    topic = state["topic"]
    groq_api_key = userdata.get('api_key')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [
                {"role": "user", "content": f"Generate one flashcard on {topic}. Format:\nQuestion: ...\nAnswer: ..."}
            ],
        }
    )

    if response.status_code != 200:
        raise Exception(f"Groq API error: {response.text}")

    content = response.json()["choices"][0]["message"]["content"]

    question = content.split("Question:")[1].split("Answer:")[0].strip()
    answer = content.split("Answer:")[1].strip()

    return {
        "flashcard": {"question": question, "answer": answer}
    }


def evaluate(state: StudyState):
    groq_api_key = userdata.get('api_key')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [
                {
                    "role": "user",
                    "content": f"""
                    Question: {state['flashcard']['question']}
                    Correct Answer: {state['flashcard']['answer']}
                    Student Answer: {state['learner_answer']}

                    Is the student answer correct? Reply only: correct or incorrect
                    """
                }
            ],
        }
    )

    if response.status_code != 200:
        raise Exception(f"Groq API error: {response.text}")

    result = response.json()["choices"][0]["message"]["content"].strip().lower()

    return {
        "answer": result,
        "progress": state["progress"] + [{
            "question": state["flashcard"]["question"],
            "learner_answer": state["learner_answer"],
            "result": result
        }]
    }


def format_output(state: StudyState):
    return {"answer": f"Result: {state['answer']}"}


# 3. Build the Graph
graph = StateGraph(StudyState)
graph.add_node("generate", generate_flashcard)
graph.add_node("evaluate", evaluate)
graph.add_node("format", format_output)

# 4. Add Edges
graph.set_entry_point("generate")
graph.add_edge("generate", "evaluate")
graph.add_edge("evaluate", "format")
graph.add_edge("format", END)

# 5. Example Run
if __name__ == "__main__":
    state = {
        "topic": "Photosynthesis",
        "learner_answer": "process by which plants make food",
        "flashcard": {},
        "answer": "",
        "progress": []
    }

    app = graph.compile()
    result = app.invoke(state)

    print(result["answer"])


Result: incorrect


In [ ]:
# Scenario: AI-Powered Project Tracker (Task-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each project:
# - task → The specific work item or milestone (e.g., "Prepare Q1 financial report").
# - status → The current state of the task (e.g., "in progress", "completed", "blocked").
# - log → A history of all updates, including who made them and when.

# 2. Workflow (Graph of States)
# Each project update flows through nodes until the task status is refreshed:
# - Input Node
# - Team member submits an update (e.g., "Report draft completed").
# - State updates: task = "Q1 financial report"
# - Processing Node (Groq API)
# - Groq interprets the update and assigns a status:
# - status = "completed"
# - Response Node
# - Assistant confirms the update back to the team:
# - "Task Q1 financial report marked as completed."
# - History Node
# - Logs the update:
# - log = [{task: "Q1 financial report", update: "draft completed", status: "completed", timestamp}]


from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict

# 1. Define State
class ProjectState(TypedDict):
    task: str
    update: str
    status: str
    log: List[Dict[str, str]]
    answer: str


# 2. Define Nodes

def process_update(state: ProjectState):
    update = state["update"].lower()

    if "complete" in update:
        status = "completed"
    elif "progress" in update:
        status = "in progress"
    elif "block" in update:
        status = "blocked"
    else:
        status = "in progress"

    return {
        "status": status
    }


def update_log(state: ProjectState):
    return {
        "log": state["log"] + [{
            "task": state["task"],
            "update": state["update"],
            "status": state["status"]
        }]
    }


def format_output(state: ProjectState):
    return {
        "answer": f"Task {state['task']} marked as {state['status']}"
    }


# 3. Build Graph
graph = StateGraph(ProjectState)

graph.add_node("process", process_update)
graph.add_node("log", update_log)
graph.add_node("format", format_output)


# 4. Add Edges
graph.set_entry_point("process")
graph.add_edge("process", "log")
graph.add_edge("log", "format")
graph.add_edge("format", END)

In [ ]:
#WITH API KEY
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
import requests
from datetime import datetime
from google.colab import userdata

# 1. Define State
class ProjectState(TypedDict):
    task: str
    update: str
    status: str
    log: List[Dict[str, str]]
    answer: str

# 2. Define Nodes

def process_update(state: ProjectState):
    groq_api_key = userdata.get("api_key")

    if not groq_api_key:
        raise ValueError("Groq API key not found")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": "llama-3.1-8b-instant",
            "messages": [
                {
                    "role": "user",
                    "content": f"""
                    Task: {state['task']}
                    Update: {state['update']}

                    Classify status as one of:
                    completed, in progress, blocked

                    Reply only with the status.
                    """
                }
            ],
        }
    )

    if response.status_code != 200:
        raise Exception(response.text)

    status = response.json()["choices"][0]["message"]["content"].strip().lower()

    return {"status": status}


def format_output(state: ProjectState):
    return {
        "answer": f"Task {state['task']} marked as {state['status']}"
    }


def update_log(state: ProjectState):
    return {
        "log": state["log"] + [{
            "task": state["task"],
            "update": state["update"],
            "status": state["status"],
            "timestamp": str(datetime.now())
        }]
    }

# 3. Build Graph
graph = StateGraph(ProjectState)
graph.add_node("process", process_update)
graph.add_node("format", format_output)
graph.add_node("log", update_log)

# 4. Add Edges
graph.set_entry_point("process")
graph.add_edge("process", "format")
graph.add_edge("format", "log")
graph.add_edge("log", END)

# 5. Run
if __name__ == "__main__":
    state = {
        "task": "Q1 financial report",
        "update": "Report draft completed",
        "status": "",
        "log": [],
        "answer": ""
    }

    app = graph.compile()
    result = app.invoke(state)

    print(result["answer"])



Task Q1 financial report marked as completed


In [ ]:
# Scenario: Customer Support Call Center
# A company runs a support chatbot that needs to route customer queries to the right department. Instead of one big script, they design a state graph where each node represents a specialized agent.

# 1. State Definition (SupportState)
# The chatbot keeps track of:
# - query → What the customer asked.
# - category → Which department it belongs to (billing, tech, general).
# - response → What the bot replies with.
# Think of this as the customer’s “ticket form.”

# 2. Router Node (route_query)
# When a customer types a question, the router scans it:
# - If the query mentions “bill” or “payment”, it routes to billing_agent.
# - If it mentions “error” or “bug”, it routes to tech_agent.
# - Otherwise, it defaults to general_agent.
# This is like a receptionist deciding which desk you should go to.

# 3. Agent Nodes
# - billing_agent → Replies with “Billing dept: [query]”.
# - tech_agent → Replies with “Tech support: [query]”.
# - general_agent → Replies with “General help: [query]”.
# Each agent specializes in its own type of problem.

# 4. Graph Flow
# - Entry point: router.
# - Router decides the path based on the query.
# - The query flows into the correct agent node.
# - The agent generates a response and ends the conversation.


from langgraph.graph import StateGraph, END
from typing import TypedDict, Literal

class SupportState(TypedDict):
    query: str
    category: str   # "billing" | "tech" | "general"
    response: str

# Router: reads state, returns next node name
def route_query(state: SupportState) -> str:
    q = state["query"].lower()
    if "bill" in q or "payment" in q:
        return "billing_agent"
    elif "error" in q or "bug" in q:
        return "tech_agent"
    else:
        return "general_agent"

def billing_agent(state):
    return {"response": "Billing dept: " + state["query"]}

def tech_agent(state):
    return {"response": "Tech support: " + state["query"]}

def general_agent(state):
    return {"response": "General help: " + state["query"]}

# Build graph with conditional routing
g = StateGraph(SupportState)
g.add_node("billing_agent", billing_agent)
g.add_node("tech_agent", tech_agent)
g.add_node("general_agent", general_agent)

# One entry point routes to 3 different nodes!
g.set_entry_point("router")
g.add_conditional_edges(
    "router",    # from node
    route_query, # function that returns next node
    {            # mapping: return value → node name
        "billing_agent":  "billing_agent",
        "tech_agent":     "tech_agent",
        "general_agent":  "general_agent",
    }
)


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class ResearchState(TypedDict):
    topic: str
    search_results: list
    analysis: str
    summary: str
    steps_done: int

# Node 1: Search for information
def search_web(state: ResearchState):
    print(f"\u001f Searching: {state['topic']}")
    # Simulate web search results
    new_results = [
        f"Article 1 about {state['topic']}",
        f"Article 2 about {state['topic']}",
    ]
    return {"search_results": state["search_results"] + new_results,
            "steps_done": state["steps_done"] + 1}

# Node 2: Analyze the results
def analyze_results(state: ResearchState):
    print(f"\u001f Analyzing {len(state['search_results'])} results")
    analysis = f"Key insights from {len(state['search_results'])} sources"
    return {"analysis": analysis,
            "steps_done": state["steps_done"] + 1}

# Node 3: Summarize
def summarize(state: ResearchState):
    print("\u001f\u001f Generating summary...")
    summary = f"Summary: {state['analysis']}"
    return {"summary": summary}

# Node 4: Check if we need more research
def should_continue(state: ResearchState) -> str:
    if len(state["search_results"]) < 3:
        return "search_web"   # Loop back!
    return END                 # Done

# Build the graph
g = StateGraph(ResearchState)
g.add_node("search_web",  search_web)
g.add_node("analyze",     analyze_results)
g.add_node("summarize",   summarize)

g.set_entry_point("search_web")
g.add_edge("search_web", "analyze")
g.add_conditional_edges("analyze", should_continue,
    {"search_web": "search_web", END: "summarize"})
g.add_edge("summarize", END)

app = g.compile()
result = app.invoke(
    {
        "topic": "Quantum Computing",
        "search_results": [],
        "analysis": "",
        "summary": "",
        "steps_done": 0
    }
)
print(result["summary"])


 Searching: Quantum Computing
 Analyzing 2 results
 Searching: Quantum Computing
 Analyzing 4 results
 Generating summary...
Summary: Key insights from 4 sources


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class ResearchState(TypedDict):
    topic: str
    search_results: list
    analysis: str
    summary: str
    steps_done: int

# 2. Helper: Groq API call
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"): # Changed model to a known working one
    groq_api_key = userdata.get('api_key')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: 'choices' key missing or empty. Full response: {response_json}")

    return response_json["choices"][0]["message"]["content"]

# 3. Nodes
def search_web(state: ResearchState):
    print(f" Searching: {state['topic']}")
    # Call Groq to generate snippets
    new_results = [
        groq_call(f"Give me a short article snippet about {state['topic']}"),
        groq_call(f"Give me another snippet about {state['topic']}")
    ]
    results = state["search_results"] + new_results
    return {
        "search_results": results,
        "steps_done": state["steps_done"] + 1
    }

def analyze_results(state: ResearchState):
    print(f" Analyzing {len(state['search_results'])} results")
    joined_results = "\n".join(state["search_results"])
    analysis = groq_call(f"Analyze these sources and extract key insights:\n{joined_results}")
    return {
        "analysis": analysis,
        "steps_done": state["steps_done"] + 1
    }

def summarize(state: ResearchState):
    print(" Generating summary...")
    summary = groq_call(f"Summarize this analysis in simple terms:\n{state['analysis']}")
    return {"summary": summary}

def should_continue(state: ResearchState) -> str:
    if len(state["search_results"]) < 3:
        return "search_web"   # Loop back until enough results
    return "summarize"        # Once we have 3+, move to summary

# 4. Build the graph
g = StateGraph(ResearchState)
g.add_node("search_web",  search_web)
g.add_node("analyze",     analyze_results)
g.add_node("summarize",   summarize)

g.set_entry_point("search_web")
g.add_edge("search_web", "analyze")
g.add_conditional_edges("analyze", should_continue,
    {"search_web": "search_web", "summarize": "summarize"})
g.add_edge("summarize", END)

# 5. Run the graph
if __name__ == "__main__":
    app = g.compile()
    result = app.invoke({
        "topic": "Quantum Computing",
        "search_results": [], "analysis": "",
        "summary": "", "steps_done": 0
    })
    print("\n Final Summary:\n", result["summary"])



 Searching: Quantum Computing
 Analyzing 2 results
 Searching: Quantum Computing
 Analyzing 4 results
 Generating summary...

 Final Summary:
 Here's a simple summary of the analysis:

**What is Quantum Computing?**
Quantum computing is a new way of processing information that's faster and more powerful than what we have today. It uses something called qubits (quantum bits) to solve problems that are too hard for regular computers.

**How does it work?**
Qubits can exist in many states at the same time, which helps us solve complex problems faster. We can use algorithms (like recipes) to help the computer solve these problems.

**What can we do with Quantum Computing?**
It can help us in many areas like medicine, finance, climate modeling, and materials science. We can also use it to break codes and find the best solutions to big problems.

**Key Benefits**
It's extremely fast, can process many things at the same time, and can find the best solutions to complex problems.

**What has be

In [ ]:
# Scenario: AI Symptom Tracker (Question-Based)
# 1. State Definition
# The assistant maintains a notebook-like state for each patient:
# - symptom → The patient’s reported issue (e.g., "persistent cough").
# - observations → Notes or snippets generated by Groq about possible causes or related conditions.
# - analysis → A synthesized interpretation of the observations.
# - recommendation → A simplified, non-medical summary suggesting next steps (e.g., "consult a doctor if symptoms persist").
# - steps_done → A counter tracking progress through the workflow.

# 2. Workflow (Graph of States)
# Each patient query flows through nodes:
# - Symptom Input Node
# - Patient reports a symptom.
# - State updates: symptom = "persistent cough"
# - Observation Node (Groq API)
# - Groq generates possible related factors or general information.
# - Updates observations.
# - Analysis Node
# - Joins observations and extracts key insights.
# - Updates analysis.
# - Conditional Node
# - If fewer than 3 observations are collected → loop back to Observation Node.
# - If 3+ observations are available → move to Recommendation Node.
# - Recommendation Node
# - Generates a simplified, non-medical recommendation (e.g., "Seek medical advice if cough lasts more than 2 weeks").
# - Updates recommendation.
# - End Node
# - Delivers the final recommendation to the patient.

from langgraph.graph import StateGraph, END
from typing import TypedDict, List
import requests
from google.colab import userdata

# 1. Define State
class SymptomState(TypedDict):
    symptom: str
    observations: List[str]
    analysis: str
    recommendation: str
    steps_done: int


# 2. Helper: Groq API call
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"):
    groq_api_key = userdata.get('api_key')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        raise Exception(response.text)

    return response.json()["choices"][0]["message"]["content"]


# 3. Nodes

def observe(state: SymptomState):
    print(f" Observing symptom: {state['symptom']}")

    new_obs = [
        groq_call(f"Give one possible cause or fact about {state['symptom']}"),
    ]

    return {
        "observations": state["observations"] + new_obs,
        "steps_done": state["steps_done"] + 1
    }


def analyze(state: SymptomState):
    print(f" Analyzing {len(state['observations'])} observations")

    joined = "\n".join(state["observations"])

    analysis = groq_call(f"Analyze these observations and give key insights:\n{joined}")

    return {
        "analysis": analysis,
        "steps_done": state["steps_done"] + 1
    }


def recommend(state: SymptomState):
    print(" Generating recommendation...")

    recommendation = groq_call(
        f"Based on this analysis:\n{state['analysis']}\nGive a simple non-medical recommendation."
    )

    return {"recommendation": recommendation}


def should_continue(state: SymptomState) -> str:
    if len(state["observations"]) < 3:
        return "observe"
    return "recommend"


# 4. Build the graph
g = StateGraph(SymptomState)

g.add_node("observe", observe)
g.add_node("analyze", analyze)
g.add_node("recommend", recommend)

g.set_entry_point("observe")

g.add_edge("observe", "analyze")

g.add_conditional_edges(
    "analyze",
    should_continue,
    {"observe": "observe", "recommend": "recommend"}
)

g.add_edge("recommend", END)


# 5. Run the graph
if __name__ == "__main__":
    app = g.compile()

    result = app.invoke({
        "symptom": "persistent cough",
        "observations": [],
        "analysis": "",
        "recommendation": "",
        "steps_done": 0
    })

    print("\n Final Recommendation:\n", result["recommendation"])



 Observing symptom: persistent cough
 Analyzing 1 observations
 Observing symptom: persistent cough
 Analyzing 2 observations
 Observing symptom: persistent cough
 Analyzing 3 observations
 Generating recommendation...

 Final Recommendation:
 A simple non-medical recommendation based on this analysis is to 'Avoid Exposure to Known Respiratory Triggers.' This could mean:

- Quitting smoking or avoiding secondhand smoke
- Reducing exposure to pollution and dust
- Wearing a mask when outdoors, especially in areas with poor air quality
- Practicing good hygiene to prevent the spread of viral infections

By taking these steps, individuals can minimize their risk of developing respiratory conditions and help manage existing ones.


In [1]:
# Human-in-the-loop(HITL)

# Scenario: AI-Assisted Email Workflow (Question-Based)
# Context
# A company deploys an AI-powered email assistant to help employees draft, review, and send professional emails.
# The workflow is modeled as a graph of states, where each email task flows through nodes until it is either approved
# and sent or rejected.

# 1. State Definition
# The assistant maintains a notebook-like state:
# - task → The subject or purpose of the email (e.g., "Q3 Report").
# - draft → The AI-generated email draft.
# - approved → A flag indicating whether the human reviewer has approved the draft.

# 2. Workflow (Graph of States)
# Each email task flows through nodes:
# - Draft Node
# - AI generates a draft email based on the task.
# - Updates draft.
# - Review Node (Interrupt)
# - Execution pauses here.
# - Human reviewer inspects the draft and decides whether to approve or reject.
# - Updates approved.
# - Send Node
# - If approved = True → Email is sent.
# - If approved = False → Email is rejected.
# - Updates task with final status (SENT or REJECTED).
# - End Node
# - Workflow completes.

# 3. Example Flow
# - Employee: "Draft an email for the Q3 Report."
# - State: task = "Q3 Report"
# - Assistant drafts:
# Dear User,
# Regarding: Q3 Report
# [AI drafted content]
# - Human reviews → Approves draft (approved = True)
# - Assistant sends → task = "SENT: Q3 Report"
# - Final Output:  Email delivered.

#  Scenario Question:
# "Imagine you are designing an AI-powered email assistant that drafts emails, pauses for human review, and
# then either sends or rejects them. How would you structure the state and workflow graph to ensure accountability
#  and human oversight in the process?"

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import requests
from google.colab import userdata

# 1. Define State
class EmailState(TypedDict):
    task: str
    draft: str
    approved: bool

# 2. Helper: Groq API call
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"):
    groq_api_key = userdata.get('api_key')
    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets. Please set 'Groq_api'.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        try:
            error_details = response.json()
        except requests.exceptions.JSONDecodeError:
            error_details = response.text
        raise Exception(f"Groq API error (Status: {response.status_code}): {error_details}")

    response_json = response.json()
    if "choices" not in response_json or not response_json["choices"]:
        raise ValueError(f"Unexpected API response format: {response_json}")

    return response_json["choices"][0]["message"]["content"]

# 3. Nodes
def draft_email(state: EmailState):
    print(f" Drafting email for task: {state['task']}")
    draft = groq_call(f"Draft a professional email regarding: {state['task']}")
    return {"draft": draft}

def human_review(state: EmailState):
    # Interrupt node: waits for human approval
    print(f" Draft ready for review:\n\n{state['draft']}\n")
    return {}  # Pauses here until human updates 'approved'

def send_email(state: EmailState):
    if state.get("approved", False):
        print(" Email approved and sent.")
        return {"task": f"SENT: {state['task']}"}
    else:
        print(" Email rejected.")
        return {"task": f"REJECTED: {state['task']}"}

# 4. Build Graph
g = StateGraph(EmailState)
g.add_node("draft", draft_email)
g.add_node("review", human_review)
g.add_node("send", send_email)

g.set_entry_point("draft")
g.add_edge("draft", "review")
g.add_edge("review", "send")
g.add_edge("send", END)

# 5. Checkpointer
checkpointer = MemorySaver()
app = g.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]  # Pause before review
)

# 6. Run Workflow
thread = {"configurable": {"thread_id": "email-1"}}

# Step 1: Draft email
app.invoke({"task": "Q3 Report", "draft": "", "approved": False}, thread)

# Step 2: Human reviews draft and resumes
app.invoke({"approved": True}, thread)


 Drafting email for task: Q3 Report
 Drafting email for task: Q3 Report


{'task': 'Q3 Report',
 'draft': "Subject: Q3 Report - Results and Key Takeaways\n\nDear [Recipient's Name],\n\nI am writing to present the Q3 Report for [Company/Department Name]. The report highlights our performance and key achievements from the third quarter (July 1 to September 30) against our set objectives.\n\n**Key Highlights:**\n\n- Revenue growth of [Percentage] compared to the same period last year, reaching [Total Revenue] for the quarter.\n- Achieved [Key Performance Indicator 1], exceeding our set target by [Percentage].\n- Successfully executed [Strategic Initiative 1], leading to a return on investment (ROI) of [Percentage].\n- Maintained a profit margin of [Percentage] despite increased operating costs.\n\n**Challenges and Opportunities:**\n\n- Identified areas of improvement in operational efficiency, which will be addressed through [Implementation Plan].\n- Favourable market conditions for expansion into new markets.\n- Increasing demand for our services/products.\n\n

In [2]:
# Scenario Question
# "Imagine you are designing an AI-powered assistant that drafts structured reports, pauses for human review,
#and then either publishes or rejects them.
#How would you structure the state and workflow graph to ensure accountability and human oversight in the process?"

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import requests
from google.colab import userdata

# 1. Define State
class ReportState(TypedDict):
    topic: str
    draft: str
    review_status: str
    feedback: str
    final_output: str
    history: List[str]

# 2. Helper: Groq API call
def groq_call(prompt: str, model: str = "llama-3.1-8b-instant"):
    groq_api_key = userdata.get('api_key')

    if not groq_api_key:
        raise ValueError("Groq API key not found in Colab secrets.")

    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {groq_api_key}"},
        json={
            "model": model,
            "messages": [{"role": "user", "content": prompt}],
        }
    )

    if response.status_code != 200:
        raise Exception(response.text)

    return response.json()["choices"][0]["message"]["content"]

# 3. Nodes

def generate_draft(state: ReportState):
    print(f" Generating report for: {state['topic']}")
    draft = groq_call(f"Write a structured report on: {state['topic']}")
    return {
        "draft": draft,
        "history": state["history"] + ["Draft generated"]
    }

def human_review(state: ReportState):
    print(f"\n Draft for review:\n\n{state['draft']}\n")
    return {}  # Pause for human input

def revise_draft(state: ReportState):
    print(" Revising draft based on feedback...")
    new_draft = groq_call(
        f"Improve this report based on feedback:\n{state['draft']}\nFeedback: {state['feedback']}"
    )
    return {
        "draft": new_draft,
        "history": state["history"] + ["Draft revised"]
    }

def publish(state: ReportState):
    print(" Report approved and published.")
    return {
        "final_output": state["draft"],
        "history": state["history"] + ["Published"]
    }

def reject(state: ReportState):
    print(" Report rejected.")
    return {
        "history": state["history"] + ["Rejected"]
    }

def decision(state: ReportState):
    if state["review_status"] == "approved":
        return "publish"
    elif state["review_status"] == "rejected":
        return "revise"
    else:
        return "reject"

# 4. Build Graph
g = StateGraph(ReportState)

g.add_node("generate", generate_draft)
g.add_node("review", human_review)
g.add_node("revise", revise_draft)
g.add_node("publish", publish)
g.add_node("reject", reject)

g.set_entry_point("generate")

g.add_edge("generate", "review")

g.add_conditional_edges(
    "review",
    decision,
    {
        "publish": "publish",
        "revise": "revise",
        "reject": "reject"
    }
)

g.add_edge("revise", "review")
g.add_edge("publish", END)
g.add_edge("reject", END)

# 5. Checkpointer
checkpointer = MemorySaver()
app = g.compile(
    checkpointer=checkpointer,
    interrupt_before=["review"]
)

# 6. Run Workflow
thread = {"configurable": {"thread_id": "report-1"}}

# Step 1: Generate draft
app.invoke({
    "topic": "AI in Healthcare",
    "draft": "",
    "review_status": "",
    "feedback": "",
    "final_output": "",
    "history": []
}, thread)

# Step 2: Human review (example: approved)
app.invoke({
    "review_status": "approved"
}, thread)


 Generating report for: AI in Healthcare
 Generating report for: AI in Healthcare


{'topic': 'AI in Healthcare',
 'draft': '**Report: Artificial Intelligence (AI) in Healthcare**\n\n**Introduction**\n\nArtificial Intelligence (AI) has revolutionized the healthcare industry by transforming the way patient data is collected, analyzed, and utilized. The integration of AI in healthcare has the potential to improve patient outcomes, enhance the quality of care, and optimize resource utilization. In this report, we will delve into the various applications of AI in healthcare, its benefits, challenges, and future prospects.\n\n**Background**\n\nOver the years, healthcare has been dealing with data overload, inefficient workflows, and a lack of personalized care. The introduction of AI has addressed these issues, enabling healthcare professionals to make informed decisions and improve patient care. With the help of machine learning algorithms, AI can analyze large datasets, identify patterns, and predict outcomes.\n\n**Applications of AI in Healthcare**\n\n1. **Predictive An